# 03 — FAISS Index & Search

Build the FAISS vector index used by the Streamlit UI to rank resumes against a
skill / job-description query.

**Inputs** (from notebook 02):
- `data/processed/embeddings.npy`
- `data/processed/resumes_with_embeddings.json`
- `data/processed/resumes_extracted.json` (full text, for the metadata export)

**Outputs** (consumed by `streamlit/search.py`):
- `resume_index.faiss` — the production FAISS index (DS3 members only)
- `member_resumes_metadata.json` — aligned metadata for the index
- `config.json` — model name, dimensions, resume count

In [1]:
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss

PROJECT_ROOT = Path("../").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

EMB_PATH       = PROCESSED_DIR / "embeddings.npy"
META_PATH      = PROCESSED_DIR / "resumes_with_embeddings.json"
EXTRACTED_PATH = PROCESSED_DIR / "resumes_extracted.json"

# Final artifacts (paths match streamlit/config.py)
FAISS_OUT      = PROJECT_ROOT / "resume_index.faiss"
METADATA_OUT   = PROJECT_ROOT / "member_resumes_metadata.json"
CONFIG_OUT     = PROJECT_ROOT / "config.json"

MODEL_NAME     = "all-MiniLM-L6-v2"

print(f"Embeddings : {EMB_PATH}")
print(f"Metadata   : {META_PATH}")
print(f"FAISS out  : {FAISS_OUT}")

Embeddings : /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/data/processed/embeddings.npy
Metadata   : /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/data/processed/resumes_with_embeddings.json
FAISS out  : /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/resume_index.faiss


## 1 — Load embeddings & metadata

In [2]:
all_embeddings = np.load(EMB_PATH)

with open(META_PATH, "r", encoding="utf-8") as f:
    all_meta = json.load(f)

with open(EXTRACTED_PATH, "r", encoding="utf-8") as f:
    all_extracted = json.load(f)

assert len(all_embeddings) == len(all_meta) == len(all_extracted), (
    f"Row count mismatch: embeddings={len(all_embeddings)}, "
    f"meta={len(all_meta)}, extracted={len(all_extracted)}"
)

print(f"Loaded {len(all_embeddings)} embeddings  (dim={all_embeddings.shape[1]})")

source_counts = {}
for m in all_meta:
    source_counts[m["source"]] = source_counts.get(m["source"], 0) + 1
for src, cnt in sorted(source_counts.items()):
    print(f"  {src:20s} : {cnt}")

Loaded 807 embeddings  (dim=384)
  ds3_board            : 46
  ds3_members          : 274
  kaggle               : 357
  train                : 130


## 2 — Filter to DS3 member resumes (production index)

The FAISS index that powers the UI should only contain DS3 member resumes —
those are the candidates recruiters are searching for.  
Reddit / Discord / Kaggle data is kept for potential fine-tuning or evaluation.

In [3]:
member_indices = [i for i, m in enumerate(all_meta) if m["source"] == "ds3_members"]

member_embeddings = all_embeddings[member_indices].copy()
member_meta       = [all_meta[i] for i in member_indices]
member_extracted  = [all_extracted[i] for i in member_indices]

print(f"DS3 member resumes for production index: {len(member_indices)}")

if len(member_indices) == 0:
    print("\n[WARNING] No DS3 member resumes found.")
    print("Make sure notebook 01 processed the DS3 PDFs and tagged them source='ds3_members'.")
    print("Falling back to ALL resumes for now so you can test the pipeline end-to-end.")
    member_embeddings = all_embeddings.copy()
    member_meta       = all_meta
    member_extracted  = all_extracted

DS3 member resumes for production index: 274


## 3 — Build the FAISS index

We use `IndexFlatIP` (inner product) with L2-normalized vectors, which is
equivalent to cosine similarity. For the expected corpus size (<1 000 resumes)
exact search is plenty fast — no need for approximate indices.

In [4]:
dimension = member_embeddings.shape[1]

faiss.normalize_L2(member_embeddings)

index = faiss.IndexFlatIP(dimension)
index.add(member_embeddings)

print(f"FAISS index built: {index.ntotal} vectors, dim={dimension}")

FAISS index built: 274 vectors, dim=384


## 4 — Test search

In [5]:
model = SentenceTransformer(MODEL_NAME)


def search(query: str, top_k: int = 10, min_score: float = 0.0):
    """Encode a query and return the top-k matching resumes."""
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, top_k)

    results = []
    for idx, score in zip(indices[0], scores[0]):
        if idx < 0 or score < min_score:
            continue
        meta = member_meta[idx]
        results.append({
            "rank": len(results) + 1,
            "filename": meta["filename"],
            "score": round(float(score), 4),
            "source": meta["source"],
            "preview": meta.get("text_preview", "")[:200],
        })
    return results


print("Search function ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Search function ready.


In [6]:
test_queries = [
    "Python, Machine Learning, Data Science",
    "React, JavaScript, full-stack web development",
    "C++, computer vision, embedded systems",
]

for q in test_queries:
    print(f"\n{'=' * 70}")
    print(f"Query: {q}")
    print(f"{'=' * 70}")
    for r in search(q, top_k=5):
        print(f"  #{r['rank']}  score={r['score']:.4f}  {r['filename']}")
        print(f"       {r['preview'][:120]}...")


Query: Python, Machine Learning, Data Science


  #1  score=0.6279  member_resume_dhruv_patel.pdf
       Tab 1
Dhruv Patel
(747) 257-7194  dhruv.patel.imp@gmail.com  Portfolio Website  GitHub  LinkedIn
EDUCATION
University of...
  #2  score=0.6203  member_resume_ryan_rulkens.pdf
       Ryan Rulkens
linkedin.com/in/ryan-rulkens-36106b237/  (925) 663-9633  ryanrulkens@gmail.com
OBJECTIVE
Motivated and ener...
  #3  score=0.6141  member_resume_daira_merino.pdf
       Daira Merino
La Jolla, CA 92092  dlmerino@ucsd.edu  661-574-5824  linkedin.com/in/dairamerino/
Summary
Data-curious UC S...
  #4  score=0.6139  member_resume_sean_harmon.pdf
       Sean Harmon
t2harmon@ucsd.edu github.com/ICharmU
I am interested in the mathematics behind machine learning techniques. ...
  #5  score=0.6099  member_resume_prisha_maiti.pdf
       Prisha Maiti
prisha.maiti@gmail.com | (858) 924-2908 | https://www.linkedin.com/in/prisha-maiti-39a80a1a4/
Looking for n...

Query: React, JavaScript, full-stack web development


  #1  score=0.4124  member_resume_van_nguyen.pdf
       Van Nguyen
van019@ucsd.edu linkedin.com/in/xinchaonnv10
EDUCATION
University of California San Diego June 2027
B.S in Co...
  #2  score=0.3916  member_resume_jeremy_chen.pdf
       Jeremy Chen
650-667-0018 | jtc011@ucsd.edu | https://www.linkedin.com/in/jeremy-t-chen/ |
https://jeremychen144.wixsite....
  #3  score=0.3875  member_resume_anusha.pdf
       Anusha Chadalavada
achadalavada@ucsd.edu | github.com/anushachad | linkedin.com/in/anushachad/
EDUCATION
University of C...
  #4  score=0.3613  member_resume_lauren_lee.pdf
       Lauren Lee
858-705-9990 | laurenayako@gmail.com | LinkedIn | GitHub
Education
UC San Diego San Diego, CA
Major: Data Sci...
  #5  score=0.3058  member_resume_gloria_lee.pdf
       Gloria Lee
San Diego gll004@ucsd.edu (408) 824-0729 linkedin github.com/Mianbao004
Education
University of California Sa...

Query: C++, computer vision, embedded systems


  #1  score=0.4063  member_resume_ray_wag.pdf
       Yirui (Ray) Wang
Phone: (858) 396-4999 | Email: rayyiruiwang@gmail.com | LinkedIn | GitHub
EDUCATION
University of Calif...
  #2  score=0.4054  member_resume_issac_roy.pdf
       Issac Roy
U.S. Citizen, San Diego, CA
(cid:128) issacroy.com  github.com/TheBoyRoy05 (cid:239) linkedin.com/in/issacroy ...
  #3  score=0.3739  member_resume_saksham_raheja.pdf
       Saksham Raheja
sakshamraheja11@gmail.com | linkedin.com/in/hashsaksham | github.com/hashsaksham
EDUCATION
University of ...
  #4  score=0.3575  member_resume_gloria_lee.pdf
       Gloria Lee
San Diego gll004@ucsd.edu (408) 824-0729 linkedin github.com/Mianbao004
Education
University of California Sa...
  #5  score=0.3537  member_resume_nik_espinoza.pdf
       Nikolas Espinoza
npespinoza@ucsd.edu | LinkedIn Account | Github Account
Education
University of California, San Diego L...


## 5 — Export artifacts for Streamlit

Write the three files that `streamlit/search.py` expects:
1. `resume_index.faiss`
2. `member_resumes_metadata.json`
3. `config.json`

In [7]:
# 1. FAISS index
faiss.write_index(index, str(FAISS_OUT))
print(f"Saved FAISS index : {FAISS_OUT}  ({FAISS_OUT.stat().st_size / 1024:.1f} KB)")

# 2. Metadata — includes full text so the UI can show previews
export_meta = []
for meta, ext in zip(member_meta, member_extracted):
    export_meta.append({
        "filename": meta["filename"],
        "file_path": meta["file_path"],
        "text": ext["text"],
        "source": meta["source"],
        **(meta.get("metadata") or {}),
    })

with open(METADATA_OUT, "w", encoding="utf-8") as f:
    json.dump(export_meta, f, indent=2, ensure_ascii=False)
print(f"Saved metadata    : {METADATA_OUT}  ({METADATA_OUT.stat().st_size / 1024:.1f} KB)")

# 3. Config
config = {
    "model_name": MODEL_NAME,
    "embedding_dimension": dimension,
    "num_resumes": index.ntotal,
}
with open(CONFIG_OUT, "w") as f:
    json.dump(config, f, indent=2)
print(f"Saved config      : {CONFIG_OUT}")

print(f"\nDone! The Streamlit app can now load these artifacts.")

Saved FAISS index : /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/resume_index.faiss  (411.0 KB)
Saved metadata    : /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/member_resumes_metadata.json  (1057.4 KB)
Saved config      : /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/config.json

Done! The Streamlit app can now load these artifacts.


## 6 — (Optional) Build a separate training index

If you want a FAISS index of the Reddit / Discord / Kaggle resumes for
evaluation or cross-corpus search, build it here.

In [8]:
# TODO: Uncomment if you want a separate training/reference index.
#
# training_indices = [i for i, m in enumerate(all_meta) if m["source"] != "ds3_members"]
#
# if training_indices:
#     training_embeddings = all_embeddings[training_indices].copy().astype("float32")
#     faiss.normalize_L2(training_embeddings)
#
#     training_index = faiss.IndexFlatIP(dimension)
#     training_index.add(training_embeddings)
#
#     TRAINING_FAISS_OUT = PROCESSED_DIR / "training_index.faiss"
#     faiss.write_index(training_index, str(TRAINING_FAISS_OUT))
#     print(f"Training index saved: {TRAINING_FAISS_OUT} ({training_index.ntotal} vectors)")
# else:
#     print("No non-DS3 resumes to build a training index from.")